# 02 — Explore the live MBTA V3 API

Exploration only: ingestion belongs in `src/`.

**API:** https://api-v3.mbta.com (JSON:API format). It works without a key at 20 requests/minute.
For more headroom, get a free key at https://api-v3.mbta.com and set the `MBTA_API_KEY` environment variable; never paste the key into this notebook.

In [ ]:
import os
from datetime import date

import pandas as pd
import requests

BASE_URL = "https://api-v3.mbta.com"
ROUTE = "1"  # a busy, frequent route to look at

session = requests.Session()
if os.environ.get("MBTA_API_KEY"):
    session.headers["x-api-key"] = os.environ["MBTA_API_KEY"]


def get(path, **params):
    resp = session.get(f"{BASE_URL}/{path}", params=params, timeout=30)
    resp.raise_for_status()
    print(path, "| rate limit remaining:", resp.headers.get("x-ratelimit-remaining"))
    return resp.json()


def flatten(items):
    """JSON:API resources -> DataFrame of id, attributes and related ids."""
    rows = []
    for item in items:
        row = {"id": item["id"], **item.get("attributes", {})}
        for name, rel in item.get("relationships", {}).items():
            data = rel.get("data")
            if isinstance(data, dict):
                row[f"{name}_id"] = data["id"]
        rows.append(row)
    return pd.DataFrame(rows)

## Bus routes (`route_type` 3)

In [ ]:
routes = flatten(get("routes", **{"filter[type]": 3})["data"])
print(len(routes), "bus routes")
routes[["id", "short_name", "long_name", "direction_names", "direction_destinations"]].head(10)

## Schedules: what's *planned* (known in advance, so safe for features)

In [ ]:
sched = flatten(get("schedules", **{"filter[route]": ROUTE, "filter[date]": date.today().isoformat()})["data"])
print(len(sched), "scheduled stop events today on route", ROUTE)
sched.head()

In [ ]:
sched[["timepoint", "pickup_type", "drop_off_type"]].apply(lambda s: s.value_counts(dropna=False)).T

## Predictions: what's expected *right now*

This is real-time data. It's useful for collecting actual departure times (labels), but it's recorded during the trip,
so it must **never** become a feature (leakage).

In [ ]:
pred_json = get("predictions", **{"filter[route]": ROUTE, "include": "schedule"})
pred = flatten(pred_json["data"])
print(len(pred), "predictions")
pred.head()

In [ ]:
pred.reindex(columns=["schedule_relationship", "status", "update_type", "revenue"]).apply(lambda s: s.value_counts(dropna=False)).T

### Predicted delay vs. schedule

Join each prediction to its included schedule. This approximates the label the pipeline would compute once the trip is over.
Empty overnight, when no buses are running.

In [ ]:
if pred.empty:
    print("No live predictions right now (no buses running?). Re-run during service hours.")
else:
    included = flatten([i for i in pred_json.get("included", []) if i["type"] == "schedule"])
    sched_cols = included[["id", "departure_time", "arrival_time"]].add_prefix("sched_")
    merged = pred.merge(sched_cols, left_on="schedule_id", right_on="sched_id", how="left")
    predicted = pd.to_datetime(merged["departure_time"].fillna(merged["arrival_time"]), utc=True)
    scheduled = pd.to_datetime(merged["sched_departure_time"].fillna(merged["sched_arrival_time"]), utc=True)
    merged["delay_s"] = (predicted - scheduled).dt.total_seconds()
    merged["is_late"] = merged["delay_s"] > 180  # strict >

    print("predictions with no schedule (added trips):", merged["sched_id"].isna().sum())
    print(merged["delay_s"].describe().round(0))
    print("share predicted late:", round(merged.loc[merged["delay_s"].notna(), "is_late"].mean(), 3))

## Vehicles: live positions (real-time, so not a feature)

In [ ]:
vehicles = flatten(get("vehicles", **{"filter[route]": ROUTE})["data"])
vehicles.reindex(columns=["id", "label", "current_status", "current_stop_sequence", "occupancy_status", "updated_at"]).head()

## Alerts: planned detours and service changes

The spec lists alerts as a feature source. Only alerts **announced before** the trip may be used, so filter on `created_at`.

In [ ]:
alerts = flatten(get("alerts", **{"filter[route_type]": 3})["data"])
print(len(alerts), "active bus alerts")
alerts.reindex(columns=["effect", "cause", "severity", "created_at", "active_period", "header"]).head()

## How the API compares to the historical CSVs

- The CSVs only cover **timepoints**. The API has every stop (`timepoint` flag on schedules).
- The CSVs have **actual** times. The API only has **predictions** until a trip finishes, so daily ingestion has to
  snapshot predictions or vehicles and keep the last observed time per stop. The API doesn't serve history.
- API times are ISO-8601 with an Eastern offset. CSV times are offsets on a dummy 1900-01-01 date.
- The rate limit (20/min without a key) matters for any loop over routes, so get a key before building ingestion.